In [ ]:
import pandas as pd
import numpy as np
from ast import literal_eval
import base64
import array
import matplotlib.pyplot as plt
import datetime
import os
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset,DataLoader
from torchsampler import ImbalancedDatasetSampler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torchvision import datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import models
from torchvision import utils
import matplotlib.pyplot as plt
%matplotlib inline
import time
import copy
import random
from sklearn import metrics
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_curve,auc
from torchsummary import summary
from tableone import TableOne
import copy
from lifelines.utils import concordance_index
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold



device='cuda'

# Model training

In [ ]:
class Swish(nn.Module):
    def __init__(self):
        super().__init__()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return x * self.sigmoid(x)

class SEBlock(nn.Module):
    def __init__(self, in_channels, r=4):
        super().__init__()

        self.squeeze = nn.AdaptiveAvgPool1d(1)
        self.excitation = nn.Sequential(
            nn.Linear(in_channels, in_channels * r),
            Swish(),
            nn.Linear(in_channels * r, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.squeeze(x)
        x = x.view(x.size(0), -1)
        x = self.excitation(x)
        x = x.view(x.size(0), x.size(1), 1)
        return x

class MBConv(nn.Module):
    expand = 6
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, se_scale=4, p=0.5):
        super().__init__()
        # first MBConv is not using stochastic depth
        self.p = torch.tensor(p).float() if (in_channels == out_channels) else torch.tensor(1).float()

        self.residual = nn.Sequential(
            nn.Conv1d(in_channels, in_channels * MBConv.expand, 1, stride=stride, padding=0, bias=False),
            nn.BatchNorm1d(in_channels * MBConv.expand, momentum=0.99, eps=1e-3),
            Swish(),
            nn.Conv1d(in_channels * MBConv.expand, in_channels * MBConv.expand, kernel_size=kernel_size,
                      stride=1, padding=kernel_size//2, bias=False, groups=in_channels*MBConv.expand),
            nn.BatchNorm1d(in_channels * MBConv.expand, momentum=0.99, eps=1e-3),
            Swish()
        )

        self.se = SEBlock(in_channels * MBConv.expand, se_scale)

        self.project = nn.Sequential(
            nn.Conv1d(in_channels*MBConv.expand, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm1d(out_channels, momentum=0.99, eps=1e-3)
        )

        self.shortcut = (stride == 1) and (in_channels == out_channels)

    def forward(self, x):
        # stochastic depth
        if self.training:
            if not torch.bernoulli(self.p):
                return x

        x_shortcut = x
        x_residual = self.residual(x)
        x_se = self.se(x_residual)

        x = x_se * x_residual
        x = self.project(x)

        if self.shortcut:
            x= x_shortcut + x

        return x

class SepConv(nn.Module):
    expand = 1
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, se_scale=4, p=0.5):
        super().__init__()
        # first SepConv is not using stochastic depth
        self.p = torch.tensor(p).float() if (in_channels == out_channels) else torch.tensor(1).float()

        self.residual = nn.Sequential(
            nn.Conv1d(in_channels * SepConv.expand, in_channels * SepConv.expand, kernel_size=kernel_size,
                      stride=1, padding=kernel_size//2, bias=False, groups=in_channels*SepConv.expand),
            nn.BatchNorm1d(in_channels * SepConv.expand, momentum=0.99, eps=1e-3),
            Swish()
        )

        self.se = SEBlock(in_channels * SepConv.expand, se_scale)

        self.project = nn.Sequential(
            nn.Conv1d(in_channels*SepConv.expand, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm1d(out_channels, momentum=0.99, eps=1e-3)
        )

        self.shortcut = (stride == 1) and (in_channels == out_channels)

    def forward(self, x):
        # stochastic depth
        if self.training:
            if not torch.bernoulli(self.p):
                return x

        x_shortcut = x
        x_residual = self.residual(x)
        x_se = self.se(x_residual)

        x = x_se * x_residual
        x = self.project(x)

        if self.shortcut:
            x= x_shortcut + x

        return x

class EfficientNet(nn.Module):
    def __init__(self, num_classes=10, width_coef=1., depth_coef=1., scale=1., dropout=0.2, se_scale=4, stochastic_depth=False, p=0.5):
        super().__init__()
        channels = [32, 16, 24, 40, 80, 112, 192, 320, 1280]
        repeats = [1, 2, 2, 3, 3, 4, 1]
        strides = [1, 2, 2, 2, 1, 2, 1]
        kernel_size = [3, 3, 5, 3, 5, 5, 3]
        depth = depth_coef
        width = width_coef

        channels = [int(x*width) for x in channels]
        repeats = [int(x*depth) for x in repeats]

        # stochastic depth
        if stochastic_depth:
            self.p = p
            self.step = (1 - 0.5) / (sum(repeats) - 1)
        else:
            self.p = 1
            self.step = 0


        # efficient net
        self.upsample = nn.Upsample(scale_factor=scale, mode='linear', align_corners=False)

        self.stage1 = nn.Sequential(
            nn.Conv1d(8, channels[0],3, stride=2, padding=1, bias=False),
            nn.BatchNorm1d(channels[0], momentum=0.99, eps=1e-3)
        )

        self.stage2 = self._make_Block(SepConv, repeats[0], channels[0], channels[1], kernel_size[0], strides[0], se_scale)

        self.stage3 = self._make_Block(MBConv, repeats[1], channels[1], channels[2], kernel_size[1], strides[1], se_scale)

        self.stage4 = self._make_Block(MBConv, repeats[2], channels[2], channels[3], kernel_size[2], strides[2], se_scale)

        self.stage5 = self._make_Block(MBConv, repeats[3], channels[3], channels[4], kernel_size[3], strides[3], se_scale)

        self.stage6 = self._make_Block(MBConv, repeats[4], channels[4], channels[5], kernel_size[4], strides[4], se_scale)

        self.stage7 = self._make_Block(MBConv, repeats[5], channels[5], channels[6], kernel_size[5], strides[5], se_scale)

        self.stage8 = self._make_Block(MBConv, repeats[6], channels[6], channels[7], kernel_size[6], strides[6], se_scale)

        self.stage9 = nn.Sequential(
            nn.Conv1d(channels[7], channels[8], 1, stride=1, bias=False),
            nn.BatchNorm1d(channels[8], momentum=0.99, eps=1e-3),
            Swish()
        ) 

        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.linear = nn.Linear(channels[8], num_classes)
        self.softmax=nn.Softmax(dim=1)

    def forward(self, x):
        x = self.upsample(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.stage5(x)
        x = self.stage6(x)
        x = self.stage7(x)
        x = self.stage8(x)
        x = self.stage9(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.linear(x)
        xsoft=self.softmax(x)
        return x,xsoft


    def _make_Block(self, block, repeats, in_channels, out_channels, kernel_size, stride, se_scale):
        strides = [stride] + [1] * (repeats - 1)
        layers = []
        for stride in strides:
            layers.append(block(in_channels, out_channels, kernel_size, stride, se_scale, self.p))
            in_channels = out_channels
            self.p -= self.step

        return nn.Sequential(*layers)


def efficientnet_b0(num_classes=2):
    return EfficientNet(num_classes=num_classes, width_coef=1.0, depth_coef=1.0, scale=1.0,dropout=0.2, se_scale=4)

def efficientnet_b1(num_classes=2):
    return EfficientNet(num_classes=num_classes, width_coef=1.0, depth_coef=1.1, scale=240/224, dropout=0.2, se_scale=4)

def efficientnet_b2(num_classes=2):
    return EfficientNet(num_classes=num_classes, width_coef=1.1, depth_coef=1.2, scale=260/224., dropout=0.3, se_scale=4)

def efficientnet_b3(num_classes=2):
    return EfficientNet(num_classes=num_classes, width_coef=1.2, depth_coef=1.4, scale=300/224, dropout=0.3, se_scale=4)

def efficientnet_b4(num_classes=2):
    return EfficientNet(num_classes=num_classes, width_coef=1.4, depth_coef=1.8, scale=380/224, dropout=0.4, se_scale=4)

def efficientnet_b5(num_classes=2):
    return EfficientNet(num_classes=num_classes, width_coef=1.6, depth_coef=2.2, scale=456/224, dropout=0.4, se_scale=4)

def efficientnet_b6(num_classes=2):
    return EfficientNet(num_classes=num_classes, width_coef=1.8, depth_coef=2.6, scale=528/224, dropout=0.5, se_scale=4)

def efficientnet_b7(num_classes=2):
    return EfficientNet(num_classes=num_classes, width_coef=2.0, depth_coef=3.1, scale=600/224, dropout=0.5, se_scale=4)

In [ ]:
filedir='' # enter file path here
wf_total=torch.load(filedir+'.pt') # enter the name of the waveform file here

In [ ]:
def pipeline(bs,lr,weight_decay,epoch_num,optimizer_type,label,trainvaltest_iter,themodel,modelname):
    
    df_merged3=pd.read_csv('.csv') # enter the name of the data file here
    print('len df_merged3',len(df_merged3))

    index_test=[]
    index_trainval=[]
    for i in range(len(df_merged3)):
        if '체크업' in df_merged3['처방코드(코드명)'][i]: # Korean term for checkup
            index_test.append(i)
        else:
            index_trainval.append(i)

    df_test_temp=df_merged3.iloc[index_test]
    df_test_temp.reset_index(inplace=True,drop=True)
    df_test_temp=df_test_temp[df_test_temp['datediff_int']<=0]
    df_test_temp=df_test_temp[df_test_temp['datediff_int']>=-0]
    df_test_temp.reset_index(inplace=True,drop=True)
    print('test length',len(df_test_temp))
    print('test >= 100:',len(df_test_temp[df_test_temp['CACS']>=100]))
    print('test >= 400:',len(df_test_temp[df_test_temp['CACS']>=400]))
    print('test >= 1000:',len(df_test_temp[df_test_temp['CACS']>=1000]))

    df_test=df_merged3.iloc[index_test]
    df_trainval=df_merged3.iloc[index_trainval]

    df_test.reset_index(inplace=True,drop=True)
    df_trainval.reset_index(inplace=True,drop=True)

    df_test=df_test[df_test['datediff_int']<=0]
    df_test=df_test[df_test['datediff_int']>=-0]
    df_test.reset_index(inplace=True,drop=True)

    df_val=df_trainval.iloc[:len(df_trainval)//5]
    df_train=df_trainval.iloc[len(df_trainval)//5:]

    df_train.reset_index(inplace=True,drop=True)
    df_val.reset_index(inplace=True,drop=True)

    print('len df_train',len(df_train))
    print('len df_val',len(df_val))
    print('len df_test',len(df_test))
    print('len df_test positive',np.sum(df_test[label]))
    
    savepath=''# enter savepath here
    
    df_train.to_csv(savepath+'df_train.csv')
    df_val.to_csv(savepath+'df_val.csv')
    df_test.to_csv(savepath+'df_test.csv')
    
    class Agatston_train(Dataset):
        def __init__(self):
            self.thedf=df_train
            self.len=len(self.thedf)

        def __getitem__(self,index):
            self.primary_key=self.thedf['primary_key'][index]
            randomint=random.randint(0,3750)
            self.input=wf_total[self.primary_key][:,randomint:randomint+1250]
            self.output=self.thedf[label][index]
            return self.input,self.output

        def get_labels(self): 
            return np.array(self.thedf[label])

        def __len__(self):
            return self.len

    class Agatston_val(Dataset):
        def __init__(self):
            self.thedf=df_val
            self.len=len(self.thedf)

        def __getitem__(self,index):
            self.primary_key=self.thedf['primary_key'][index]
            randomint=random.randint(0,3750)
            self.input=wf_total[self.primary_key][:,randomint:randomint+1250]
            self.output=self.thedf[label][index]
            return self.input,self.output

        def get_labels(self): 
            return np.array(self.thedf[label])

        def __len__(self):
            return self.len

    class Agatston_test(Dataset):
        def __init__(self):
            self.thedf=df_test
            self.len=len(self.thedf)*4

        def __getitem__(self,index):
            index_a=index//4
            index_b=index%4
            self.primary_key=self.thedf['primary_key'][index_a]
            self.input=wf_total[self.primary_key][:,index_b*1250:(index_b+1)*1250]
            self.output=self.thedf[label][index_a]
            return self.input,self.output

        def __len__(self):
            return self.len
    

    train_dataset = Agatston_train()
    val_dataset = Agatston_val()
    test_dataset = Agatston_test()

    train_dataset_dataloader = DataLoader(train_dataset,batch_size=bs, sampler=ImbalancedDatasetSampler(train_dataset))
    val_dataset_dataloader = DataLoader(val_dataset,batch_size=bs)
    test_dataset_dataloader = DataLoader(test_dataset,batch_size=bs)
    

    model=themodel
    criterion=nn.CrossEntropyLoss().to(device)
    if optimizer_type=='Adam':
        optimizer=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=weight_decay)
    elif optimizer_type=='RMSprop':
        optimizer=torch.optim.RMSprop(model.parameters(),lr=lr, weight_decay=weight_decay,alpha=0.9, momentum=0.1)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,patience=5,factor=0.5,mode='max')


    targets_test=[]
    prediction_test=[]

    for epoch in range(epoch_num):
        now1=datetime.datetime.now()
        for phase in trainvaltest_iter:
            if phase=='training':
                running_loss=0.0
                acc=0.
                correct=0
                for param_group in optimizer.param_groups:
                    print(param_group['lr']) 
                TP=0
                TN=0
                FP=0
                FN=0
                model.train()
                for i,data in enumerate(train_dataset_dataloader):
                    inputs,targets=data
                    inputs=inputs.to(device)
                    targets=targets.long().to(device)
                    optimizer.zero_grad()
                    outputs,outputssoft=model(inputs)
                    loss=criterion(outputs,targets)
                    loss.backward()
                    optimizer.step()
                    running_loss+=loss.item()
                    prediction=torch.max(outputs.data,1)[1]
                    correct+=prediction.eq(targets.data.view_as(prediction)).cpu().sum()
                    for a,b in zip(targets,prediction):
                        if a==1 and b==1:
                            TP+=1
                        elif a==1 and b==0:
                            FN+=1
                        elif a==0 and b==1:
                            FP+=1
                        else:
                            TN+=1 
                sensitivity=(TP/(TP+FN))
                specificity=(TN/(TN+FP))
                try:
                    precision=(TP/(TP+FP))
                except:
                    precision=0
                if precision!=0:
                    f1score=((2*precision*sensitivity)/(precision+sensitivity))
                else:
                    f1score=0
                print("[%d] loss: %.3f  sensitivity: %.3f specificity: %.3f precision: %.3f f1score: %.3f" % (epoch+1,running_loss/100,sensitivity,specificity,precision,f1score))     


            if phase=='val':
                model.eval()
                running_loss=0.0
                acc=0.
                correct=0
                TP=0
                TN=0
                FP=0
                FN=0
                prediction_score=[]
                targets_list=[]
                with torch.no_grad():
                    for i,data in enumerate(val_dataset_dataloader):
                        inputs,targets=data
                        for j in range(len(targets)):
                            targets_list.append(targets[j])
                        inputs=inputs.to(device)
                        targets=targets.long().to(device)
                        outputs,outputssoft=model(inputs)
                        loss=criterion(outputs,targets)
                        running_loss+=loss.item()
                        prediction=torch.max(outputs.data,1)[1]
                        correct+=prediction.eq(targets.data.view_as(prediction)).cpu().sum()

                        for a,b in zip(targets,prediction):
                            if a==1 and b==1:
                                TP+=1
                            elif a==1 and b==0:
                                FN+=1
                            elif a==0 and b==1:
                                FP+=1
                            else:
                                TN+=1
                        for j in range(len(outputssoft)):
                            prediction_score.append(outputssoft[j][1].item())

                try:
                    sensitivity=(TP/(TP+FN))
                except:
                    sensitivity=0.5
                try:
                    specificity=(TN/(TN+FP))
                except:
                    specificity=0.5
                try:
                    precision=(TP/(TP+FP))
                except:
                    precision=0
                if precision!=0:
                    f1score=((2*precision*sensitivity)/(precision+sensitivity))
                else:
                    f1score=0                

                fpr_temp,tpr_temp,threshold=metrics.roc_curve(targets_list,prediction_score,pos_label=1)
                AUROC_val=metrics.auc(fpr_temp,tpr_temp)
                AP=average_precision_score(targets_list,prediction_score)

                print("[%d] loss: %.3f AUROC: %.3f AUPRC: %.3f sensitivity: %.3f specificity: %.3f precision: %.3f f1score: %.3f" % (epoch+1,running_loss/100,AUROC_val,AP, sensitivity,specificity,precision,f1score))     

            if phase=='test':
                model.eval()
                running_loss=0.0
                acc=0.
                correct=0
                TP=0
                TN=0
                FP=0
                FN=0
                prediction_score=[]
                targets_list=[]
                with torch.no_grad():
                    for i,data in enumerate(test_dataset_dataloader):
                        inputs,targets=data
                        for j in range(len(targets)):
                            targets_list.append(targets[j])
                        inputs=inputs.to(device)
                        targets=targets.long().to(device)
                        outputs,outputssoft=model(inputs)
                        loss=criterion(outputs,targets)
                        running_loss+=loss.item()
                        prediction=torch.max(outputs.data,1)[1]
                        correct+=prediction.eq(targets.data.view_as(prediction)).cpu().sum()

                        for a,b in zip(targets,prediction):
                            if a==1 and b==1:
                                TP+=1
                            elif a==1 and b==0:
                                FN+=1
                            elif a==0 and b==1:
                                FP+=1
                            else:
                                TN+=1
                        for j in range(len(outputssoft)):
                            prediction_score.append(outputssoft[j][1].item())

                try:
                    sensitivity=(TP/(TP+FN))
                except:
                    sensitivity=0.5
                try:
                    specificity=(TN/(TN+FP))
                except:
                    specificity=0.5
                try:
                    precision=(TP/(TP+FP))
                except:
                    precision=0
                if precision!=0:
                    f1score=((2*precision*sensitivity)/(precision+sensitivity))
                else:
                    f1score=0                
                
                targets_test=targets_list
                prediction_test=prediction_score
                targets_test=np.array(targets_test)
                prediction_test=np.array(prediction_test)
                np.save('targets.npy',targets_test)
                np.save('prediction.npy',prediction_test)
                
                fpr_temp,tpr_temp,threshold=metrics.roc_curve(targets_list,prediction_score,pos_label=1)
                AUROC=metrics.auc(fpr_temp,tpr_temp)
                AP=average_precision_score(targets_list,prediction_score)

                print("[%d] loss: %.3f AUROC: %.3f AUPRC: %.3f sensitivity: %.3f specificity: %.3f precision: %.3f f1score: %.3f" % (epoch+1,running_loss/100,AUROC,AP, sensitivity,specificity,precision,f1score))     
 
                now = datetime.datetime.now()
                nowStr2 = "{:%Y%m%d%H%M%S}".format(now)
                modelname2=nowStr2+'_'
                modelname2+=(label+'_')
                modelname2+=modelname
                modelname2+=('_'+'epoch_'+str(epoch)+'_')
                modelname2+=('AUROC_'+str(round(AUROC_val,3))+'_')
                modelname2+=('AUROC_'+str(round(AUROC,3)))
                modelname2+='.pt'

                torch.save(model.state_dict(),savepath+modelname2)

        now2=datetime.datetime.now()
        print(now2-now1)

In [ ]:
for model in [efficientnetb0]:
    for batchsize in [128,256,512]:
        for learningrate in [0.001,0.01,0.1]:
            for weightdecay in [0,0.001,0.01]:
                print('===================================================')
                print(model,batchsize,learningrate,weightdecay,exchangethres,exchangethres2)
                themodel=model().to(device)
                modelname=(str(model).split('function')[1][1:].split('at')[0][:-1]+'_')
                pipeline(batchsize,learningrate,weightdecay,50,'Adam',"label0",phases,themodel,modelname)


# Analysis

In [ ]:
# load inference outputs only
df_test=pd.read_csv('20240701_test_inference.csv')
YSH_df=pd.read_csv('20240701_YSH_inference.csv')
AUMC_df=pd.read_csv('20240701_AUMC_inference.csv')
df_val=pd.read_csv('20240701_val_inference.csv')

label0=[]
for i in range(len(df_test)):
    if df_test['CACS'][i]>0:
        label0.append(1)
    else:
        label0.append(0)
df_test['label0']=label0

label0=[]
for i in range(len(df_val)):
    if df_val['CACS'][i]>0:
        label0.append(1)
    else:
        label0.append(0)
df_val['label0']=label0

label0=[]
for i in range(len(df_YSH)):
    if df_YSH['CACS'][i]>0:
        label0.append(1)
    else:
        label0.append(0)
df_YSH['label0']=label0

label0=[]
for i in range(len(df_AUMC)):
    if df_AUMC['score_float'][i]>0:
        label0.append(1)
    else:
        label0.append(0)
df_AUMC['label0']=label0

In [ ]:
AIpredictiontotallist=[]
for i in range(25):
    AIpredictiontotallist.append('AI_prediction_'+str(i))
    
df_test['AI_average']=df_test[AIpredictiontotallist].mean(axis=1)
df_YSH['AI_average']=df_YSH[AIpredictiontotallist].mean(axis=1)
df_AUMC['AI_average']=df_AUMC[AIpredictiontotallist].mean(axis=1)
df_val['AI_average']=df_val[AIpredictiontotallist].mean(axis=1)

print('==============')
print('df val')
fpr_val,tpr_val,thresholds=metrics.roc_curve(df_val['label400'],df_val['AI_average'],pos_label=1)
AUROC_val=metrics.auc(fpr_val,tpr_val)
print(AUROC_val)

precision_val, recall_val, _ = metrics.precision_recall_curve(df_val['label400'], df_val['AI_average'], pos_label=1)
AUPRC_val = metrics.auc(recall_val, precision_val)
print("AUPRC_val:", AUPRC_val)

youden_j = tpr_val - fpr_val
max_j_index = np.argmax(youden_j)
optimal_threshold = thresholds[max_j_index]


print('========================')
print('df test')
print('=====')
print('label400')
fpr_test,tpr_test,threshold=metrics.roc_curve(df_test['label400'],df_test['AI_average'],pos_label=1)
AUROC_test=metrics.auc(fpr_test,tpr_test)
print(AUROC_test)

precision_test, recall_test, _ = metrics.precision_recall_curve(df_test['label400'], df_test['AI_average'], pos_label=1)
AUPRC_test = metrics.auc(recall_test, precision_test)
print("AUPRC_test:", AUPRC_test)

df_test['predicted'] = (df_test['AI_average'] >= optimal_threshold).astype(int)
tn, fp, fn, tp = metrics.confusion_matrix(df_test['label400'], df_test['predicted']).ravel()
accuracy = metrics.accuracy_score(df_test['label400'], df_test['predicted'])
sensitivity = tp / (tp + fn)  # also known as recall
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)  # positive predictive value (precision)
npv = tn / (tn + fn)  # negative predictive value
f1_score = metrics.f1_score(df_test['label400'], df_test['predicted'])
print(f"Accuracy: {accuracy}")
print(f"Sensitivity: {sensitivity}")
print(f"Specificity: {specificity}")
print(f"PPV (Precision): {ppv}")
print(f"NPV: {npv}")
print(f"F1 Score: {f1_score}")
print('=====')
print('label0')
fpr_test,tpr_test,threshold=metrics.roc_curve(df_test['label0'],df_test['AI_average'],pos_label=1)
AUROC_test=metrics.auc(fpr_test,tpr_test)
print(AUROC_test)

precision_test, recall_test, _ = metrics.precision_recall_curve(df_test['label0'], df_test['AI_average'], pos_label=1)
AUPRC_test = metrics.auc(recall_test, precision_test)
print("AUPRC_test:", AUPRC_test)

df_test['predicted'] = (df_test['AI_average'] >= optimal_threshold).astype(int)
tn, fp, fn, tp = metrics.confusion_matrix(df_test['label0'], df_test['predicted']).ravel()
accuracy = metrics.accuracy_score(df_test['label0'], df_test['predicted'])
sensitivity = tp / (tp + fn)  # also known as recall
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)  # positive predictive value (precision)
npv = tn / (tn + fn)  # negative predictive value
f1_score = metrics.f1_score(df_test['label0'], df_test['predicted'])
print(f"Accuracy: {accuracy}")
print(f"Sensitivity: {sensitivity}")
print(f"Specificity: {specificity}")
print(f"PPV (Precision): {ppv}")
print(f"NPV: {npv}")
print(f"F1 Score: {f1_score}")



print('========================')
print('df YSH')
print('=====')
print('label400')
fpr_YSH,tpr_YSH,threshold=metrics.roc_curve(df_YSH['label400'],df_YSH['AI_average'],pos_label=1)
AUROC_YSH=metrics.auc(fpr_YSH,tpr_YSH)
print(AUROC_YSH)

precision_YSH, recall_YSH, _ = metrics.precision_recall_curve(df_YSH['label400'], df_YSH['AI_average'], pos_label=1)
AUPRC_YSH = metrics.auc(recall_YSH, precision_YSH)
print("AUPRC_YSH:", AUPRC_YSH)

df_YSH['predicted'] = (df_YSH['AI_average'] >= optimal_threshold).astype(int)
tn, fp, fn, tp = metrics.confusion_matrix(df_YSH['label400'], df_YSH['predicted']).ravel()
accuracy = metrics.accuracy_score(df_YSH['label400'], df_YSH['predicted'])
sensitivity = tp / (tp + fn)  # also known as recall
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)  # positive predictive value (precision)
npv = tn / (tn + fn)  # negative predictive value
f1_score = metrics.f1_score(df_YSH['label400'], df_YSH['predicted'])
print(f"Accuracy: {accuracy}")
print(f"Sensitivity: {sensitivity}")
print(f"Specificity: {specificity}")
print(f"PPV (Precision): {ppv}")
print(f"NPV: {npv}")
print(f"F1 Score: {f1_score}")


print('=====')
print('label0')
fpr_YSH,tpr_YSH,threshold=metrics.roc_curve(df_YSH['label0'],df_YSH['AI_average'],pos_label=1)
AUROC_YSH=metrics.auc(fpr_YSH,tpr_YSH)
print(AUROC_YSH)

precision_YSH, recall_YSH, _ = metrics.precision_recall_curve(df_YSH['label0'], df_YSH['AI_average'], pos_label=1)
AUPRC_YSH = metrics.auc(recall_YSH, precision_YSH)
print("AUPRC_YSH:", AUPRC_YSH)

df_YSH['predicted'] = (df_YSH['AI_average'] >= optimal_threshold).astype(int)
tn, fp, fn, tp = metrics.confusion_matrix(df_YSH['label0'], df_YSH['predicted']).ravel()
accuracy = metrics.accuracy_score(df_YSH['label0'], df_YSH['predicted'])
sensitivity = tp / (tp + fn)  # also known as recall
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)  # positive predictive value (precision)
npv = tn / (tn + fn)  # negative predictive value
f1_score = metrics.f1_score(df_YSH['label0'], df_YSH['predicted'])
print(f"Accuracy: {accuracy}")
print(f"Sensitivity: {sensitivity}")
print(f"Specificity: {specificity}")
print(f"PPV (Precision): {ppv}")
print(f"NPV: {npv}")
print(f"F1 Score: {f1_score}")

print('========================')
print('df AUMC')
print('=====')
print('label400')
fpr_AUMC,tpr_AUMC,threshold=metrics.roc_curve(df_AUMC['label400'],df_AUMC['AI_average'],pos_label=1)
AUROC_AUMC=metrics.auc(fpr_AUMC,tpr_AUMC)
print(AUROC_AUMC)

precision_AUMC, recall_AUMC, _ = metrics.precision_recall_curve(df_AUMC['label400'], df_AUMC['AI_average'], pos_label=1)
AUPRC_AUMC = metrics.auc(recall_AUMC, precision_AUMC)
print("AUPRC_AUMC:", AUPRC_AUMC)

df_AUMC['predicted'] = (df_AUMC['AI_average'] >= optimal_threshold).astype(int)
tn, fp, fn, tp = metrics.confusion_matrix(df_AUMC['label400'], df_AUMC['predicted']).ravel()
accuracy = metrics.accuracy_score(df_AUMC['label400'], df_AUMC['predicted'])
sensitivity = tp / (tp + fn)  # also known as recall
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)  # positive predictive value (precision)
npv = tn / (tn + fn)  # negative predictive value
f1_score = metrics.f1_score(df_AUMC['label400'], df_AUMC['predicted'])
print(f"Accuracy: {accuracy}")
print(f"Sensitivity: {sensitivity}")
print(f"Specificity: {specificity}")
print(f"PPV (Precision): {ppv}")
print(f"NPV: {npv}")
print(f"F1 Score: {f1_score}")

print('=====')
print('label0')
fpr_AUMC,tpr_AUMC,threshold=metrics.roc_curve(df_AUMC['label0'],df_AUMC['AI_average'],pos_label=1)
AUROC_AUMC=metrics.auc(fpr_AUMC,tpr_AUMC)
print(AUROC_AUMC)

precision_AUMC, recall_AUMC, _ = metrics.precision_recall_curve(df_AUMC['label0'], df_AUMC['AI_average'], pos_label=1)
AUPRC_AUMC = metrics.auc(recall_AUMC, precision_AUMC)
print("AUPRC_AUMC:", AUPRC_AUMC)

df_AUMC['predicted'] = (df_AUMC['AI_average'] >= optimal_threshold).astype(int)
tn, fp, fn, tp = metrics.confusion_matrix(df_AUMC['label0'], df_AUMC['predicted']).ravel()
accuracy = metrics.accuracy_score(df_AUMC['label0'], df_AUMC['predicted'])
sensitivity = tp / (tp + fn)  # also known as recall
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)  # positive predictive value (precision)
npv = tn / (tn + fn)  # negative predictive value
f1_score = metrics.f1_score(df_AUMC['label0'], df_AUMC['predicted'])
print(f"Accuracy: {accuracy}")
print(f"Sensitivity: {sensitivity}")
print(f"Specificity: {specificity}")
print(f"PPV (Precision): {ppv}")
print(f"NPV: {npv}")
print(f"F1 Score: {f1_score}")

In [ ]:
AUROCs=[]
for i in range(25):
    fpr_test,tpr_test,threshold=metrics.roc_curve(df_test['label400'],df_test['AI_prediction_'+str(i)],pos_label=1)
    AUROC_test=metrics.auc(fpr_test,tpr_test)
    AUROCs.append(AUROC_test)

In [ ]:
print(np.mean(AUROCs))
print(np.std(AUROCs))

## ROC curves

In [ ]:
plt.figure(figsize=(10,10))
plt.xlim([-0.00, 1.00])
plt.ylim([-0.00, 1.00])
plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='grey',alpha=.8)


plt.plot(fpr_test, tpr_test, color='blue',label='Test AUROC %0.3f' % (AUROC_test) ,lw=1, alpha=1)
plt.plot(fpr_YSH, tpr_YSH, color='red',label=r'External validation (YSH) AUROC %0.3f' % (AUROC_YSH) ,lw=1, alpha=1)
plt.plot(fpr_AUMC, tpr_AUMC, color='green',label=r'External validation (AUMC) AUROC %0.3f' % (AUROC_AUMC) ,lw=1, alpha=1)
plt.xlabel('1-Specificity', fontsize=17)
plt.ylabel('Sensitivity', fontsize=17)
plt.title('ROC curves', fontsize=19)
#plt.legend(loc="lower right", fontsize=29)
plt.xticks(np.arange(0.2, 1.2, step=0.2))
plt.yticks(np.arange(0, 1.2, step=0.2))
plt.xticks(fontsize =15)
plt.yticks(fontsize =15)
leg=plt.legend(loc="lower right", fontsize=17)
for legobj in leg.legendHandles:
    legobj.set_linewidth(2.0)
#plt.show()

plt.savefig('figures/20240702_ROC400.png',transparent=True)

In [ ]:
plt.figure(figsize=(10,10))
plt.xlim([-0.00, 1.00])
plt.ylim([-0.00, 1.00])
plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='grey',alpha=.8)


plt.plot(fpr_test, tpr_test, color='blue',label='Test AUROC %0.3f' % (AUROC_test) ,lw=1, alpha=1)
plt.plot(fpr_YSH, tpr_YSH, color='red',label=r'External validation (YSH) AUROC %0.3f' % (AUROC_YSH) ,lw=1, alpha=1)
plt.plot(fpr_AUMC, tpr_AUMC, color='green',label=r'External validation (AUMC) AUROC %0.3f' % (AUROC_AUMC) ,lw=1, alpha=1)
plt.xlabel('1-Specificity', fontsize=17)
plt.ylabel('Sensitivity', fontsize=17)
plt.title('ROC curves', fontsize=19)
#plt.legend(loc="lower right", fontsize=29)
plt.xticks(np.arange(0.2, 1.2, step=0.2))
plt.yticks(np.arange(0, 1.2, step=0.2))
plt.xticks(fontsize =15)
plt.yticks(fontsize =15)
leg=plt.legend(loc="lower right", fontsize=17)
for legobj in leg.legendHandles:
    legobj.set_linewidth(2.0)
#plt.show()

plt.savefig('figures/20240702_ROC_CAC0.png',transparent=True)

## Boxplot of AI average by CACS group

In [ ]:
bins = [-0.1, 0.1, 100, 400, 1000, float('inf')]  # -0.1 to include 0 as a separate bin
labels = ['0', '0-100', '100-400', '400-1000', '1000+']

# Create a new column for the CACS group
df_test['CACS_group'] = pd.cut(df_test['CACS'], bins=bins, labels=labels, right=False)

# Create the boxplot with customized outlier dot size, color, and transparency
plt.figure(figsize=(20, 12))
boxplot = df_test.boxplot(column='AI_average', by='CACS_group', grid=False, 
                     flierprops=dict(marker='o', color='grey', alpha=0.6, markersize=0.6))

# Customize the plot
plt.title(' ', fontsize=13)
plt.suptitle('') 
plt.xlabel('CAC score group', fontsize=11)
plt.ylabel('AI-enabled ECG model prediction score', fontsize=11)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

# Show the plot
plt.savefig('figures/20240702_Boxplot_of_AI_average_by_CACS_group.png',transparent=True)

# PCE calculation

In [ ]:
def AHA_calculator(Race, Sex, age, total_chol, hdl_chol, sbp, on_hypertension_meds, smoker, diabetes):
    
    # Selecting coefficients based on race and sex
    if Race == 1 and Sex == 0:
        C_Age = -29.799
        C_Sq_Age = 4.884
        C_Total_Chol = 13.54
        C_Age_Total_Chol = -3.114
        C_HDL_Chol = -13.578
        C_Age_HDL_Chol = 3.149
        C_On_HypertensionMeds = 2.019
        C_Age_On_HypertensionMeds = 0
        C_Off_HypertensionMeds = 1.957
        C_Age_Off_HypertensionMeds = 0
        C_Smoker = 7.574
        C_Age_Smoker = -1.665
        C_Diabetes = 0.661
        S10 = 0.9665
        Mean_Terms = -29.18
    
    elif Race == 1 and Sex == 1:
        C_Age = 12.344
        C_Sq_Age = 0
        C_Total_Chol = 11.853
        C_Age_Total_Chol = -2.664
        C_HDL_Chol = -7.99
        C_Age_HDL_Chol = 1.769
        C_On_HypertensionMeds = 1.797
        C_Age_On_HypertensionMeds = 0
        C_Off_HypertensionMeds = 1.764
        C_Age_Off_HypertensionMeds = 0
        C_Smoker = 7.837
        C_Age_Smoker = -1.795
        C_Diabetes = 0.658
        S10 = 0.9144
        Mean_Terms = 61.18
    
    elif Race == 0 and Sex == 0:
        C_Age = 17.114
        C_Sq_Age = 0
        C_Total_Chol = 0.94
        C_Age_Total_Chol = 0
        C_HDL_Chol = -18.92
        C_Age_HDL_Chol = 4.475
        C_On_HypertensionMeds = 29.291
        C_Age_On_HypertensionMeds = -6.432
        C_Off_HypertensionMeds = 27.82
        C_Age_Off_HypertensionMeds = -6.087
        C_Smoker = 0.691
        C_Age_Smoker = 0
        C_Diabetes = 0.874
        S10 = 0.9533
        Mean_Terms = 86.61
    
    elif Race == 0 and Sex == 1:
        C_Age = 2.469
        C_Sq_Age = 0
        C_Total_Chol = 0.302
        C_Age_Total_Chol = 0
        C_HDL_Chol = -0.307
        C_Age_HDL_Chol = 0
        C_On_HypertensionMeds = 1.916
        C_Age_On_HypertensionMeds = 0
        C_Off_HypertensionMeds = 1.809
        C_Age_Off_HypertensionMeds = 0
        C_Smoker = 0.549
        C_Age_Smoker = 0
        C_Diabetes = 0.645
        S10 = 0.8954
        Mean_Terms = 19.54
    else:
        print('wrong')
        
    #risk calculation(%)
    terms = (C_Age * math.log(age)) + \
        (C_Sq_Age * math.pow(math.log(age), 2)) + \
        (C_Total_Chol * math.log(total_chol)) + \
        (C_Age_Total_Chol * math.log(age) * math.log(total_chol)) + \
        (C_HDL_Chol * math.log(hdl_chol)) + \
        (C_Age_HDL_Chol * math.log(age) * math.log(hdl_chol)) + \
        (on_hypertension_meds * C_On_HypertensionMeds * math.log(sbp)) + \
        (on_hypertension_meds * C_Age_On_HypertensionMeds * math.log(age) * math.log(sbp)) + \
        ((1 - on_hypertension_meds) * C_Off_HypertensionMeds * math.log(sbp)) + \
        ((1 - on_hypertension_meds) * C_Age_Off_HypertensionMeds * math.log(age) * math.log(sbp)) + \
        (C_Smoker * smoker) + \
        (C_Age_Smoker * math.log(age) * smoker) + \
        (C_Diabetes * diabetes)

    risk = 100 * (1 - math.pow(S10,(math.exp(terms - Mean_Terms))))
    risk=round(risk, 1)
    return risk


sexint=[]
for i in range(len(df)):
    if df['sex'][i]=='M':
        sexint.append(1)
    else:
        sexint.append(0)
df['sexint']=sexint

import math
PCE=[]
for i in range(len(df)):
    score=AHA_calculator(1,df['sexint'][i],df['year_difference'][i],df['Chol'][i],df['HDL'][i],df['SBP'][i],df['HTN'][i],df['smoking'][i],df['DM'][i]   )
    PCE.append(score)
    if np.isnan(score)==True:
        print(df.iloc[i])
df['PCE']=PCE

In [ ]:
PCEcat=[]
for i in range(len(df)):
    if df['PCE'][i]<7.5:
        PCEcat.append(1)
    elif df['PCE'][i]<20:
        PCEcat.append(2)
    else:
        PCEcat.append(3)
df['PCEcat']=PCEcat

## Survival analysis

In [ ]:
dateuntil=3650

df=pd.read_csv('20240701_survivaldata.csv',encoding='cp949')

df['checkup_date'] = pd.to_datetime(df['checkup_date'])
df['date2'] = pd.to_datetime(df['date2'])

df['status'] = df['date2'].notnull().astype(int)

df['day_difference'] = df.apply(lambda row: (row['date2'] - row['checkup_date']).days if row['status'] == 1 else (pd.Timestamp('2024-08-10') - row['checkup_date']).days, axis=1)

status_MACEonly=[]
for i in range(len(df)):
    if (df['ICD'][i]>=21 and df['ICD'][i]<=25) or df['ICD'][i]==63 or df['ICD'][i]==64:
        status_MACEonly.append(1)
    else:
        status_MACEonly.append(0)
df['status_MACEonly']=status_MACEonly


In [ ]:
import pandas as pd

# Assuming df is already loaded and the columns are prepared as per the previous code
# Calculate person-time in years
df['person_years'] = df['day_difference'] / 365.25

# Calculate total number of events
total_events = df['status'].sum()

# Calculate total person-time
total_person_time = df['person_years'].sum()

# Calculate incidence rate per 1000 person-years
incidence_rate = (total_events / total_person_time) * 1000

print(f"Incidence rate per 1000 person-years: {incidence_rate}")

In [ ]:
print(np.percentile(df['person_years'],25))
print(np.percentile(df['person_years'],50))
print(np.percentile(df['person_years'],75))

In [ ]:
thres1= #enter threshold for low vs mod/high risk
thres2= #enter threshold for low/mod vs high risk
def kmcurve(df):
    TopTertile = df[df['AI_prediction_average'] >thres2]
    MiddleTertile= df[df['AI_prediction_average'] <=thres2]
    MiddleTertile= MiddleTertile[MiddleTertile['AI_prediction_average'] >thres1]
    BottomTertile = df[df['AI_prediction_average'] <=thres1]

    kmf = KaplanMeierFitter()
    plt.figure(figsize=(10, 8))

    kmf.fit(TopTertile['day_difference'], TopTertile['status'])
    ci_lower = 1 - kmf.confidence_interval_['KM_estimate_upper_0.95']
    ci_upper = 1 - kmf.confidence_interval_['KM_estimate_lower_0.95']
    plt.fill_between(kmf.survival_function_.index, ci_lower, ci_upper, color='red',alpha=0.3)
    plt.step(kmf.survival_function_.index, 1 - kmf.survival_function_, color='red',label='High risk (AI)')

    kmf.fit(MiddleTertile['day_difference'], MiddleTertile['status'])
    ci_lower = 1 - kmf.confidence_interval_['KM_estimate_upper_0.95']
    ci_upper = 1 - kmf.confidence_interval_['KM_estimate_lower_0.95']
    plt.fill_between(kmf.survival_function_.index, ci_lower, ci_upper, color='green',alpha=0.3)
    plt.step(kmf.survival_function_.index, 1 - kmf.survival_function_, color='green',label='Moderate risk (AI)')


    kmf.fit(BottomTertile['day_difference'], BottomTertile['status'])
    ci_lower = 1 - kmf.confidence_interval_['KM_estimate_upper_0.95']
    ci_upper = 1 - kmf.confidence_interval_['KM_estimate_lower_0.95']
    plt.fill_between(kmf.survival_function_.index, ci_lower, ci_upper, color='blue',alpha=0.3)
    plt.step(kmf.survival_function_.index, 1 - kmf.survival_function_, color='blue',label='Low risk (AI)')

    plt.xlim(0, dateuntil)  # Set x-axis range
    plt.ylim([0, 0.048])  # Adjust the limits as per your data

    xlabel_size = 22  # Adjust as needed
    ylabel_size = 22  # Adjust as needed
    ticks_size = 20   # Adjust as needed
    title_size = 23   # Adjust as needed
    legend_size = 18  # Adjust as needed

    plt.xlabel('Time (days)', fontsize=xlabel_size)
    plt.ylabel('Cumulative Incidence', fontsize=ylabel_size)
    plt.xticks(np.arange(0, dateuntil, step=500), fontsize=ticks_size)  # Adjust step as per your data
    plt.yticks(np.arange(0, 0.048, step=0.005), fontsize=ticks_size)

    leg=plt.legend(loc='upper left', fontsize=legend_size)
    for legobj in leg.legendHandles:
        legobj.set_linewidth(2.5)


    plt.savefig('figures/20240709_survival_tertile.png',transparent=True)

    # Perform the multivariate logrank test
    T = pd.concat([BottomTertile['day_difference'], MiddleTertile['day_difference'], TopTertile['day_difference']])
    E = pd.concat([BottomTertile['status'], MiddleTertile['status'], TopTertile['status']])
    groups = ['Bottom'] * len(BottomTertile) + ['Middle'] * len(MiddleTertile) + ['Top'] * len(TopTertile)

    result = multivariate_logrank_test(T, groups, E)
    print(result.print_summary())
    print(result.p_value)


    T1=BottomTertile['day_difference']
    T2=MiddleTertile['day_difference']
    E1=BottomTertile['status']
    E2=MiddleTertile['status']
    logrankresults=logrank_test(T1,T2,event_observed_A=E1,event_observed_B=E2)
    print(logrankresults.print_summary())
    print(logrankresults.p_value)

    T1=MiddleTertile['day_difference']
    T2=TopTertile['day_difference']
    E1=MiddleTertile['status']
    E2=TopTertile['status']
    logrankresults=logrank_test(T1,T2,event_observed_A=E1,event_observed_B=E2)
    print(logrankresults.print_summary())
    print(logrankresults.p_value)

    df['person_years'] = df['day_difference'] / 365.25
    total_events = df['status'].sum()
    total_person_time = df['person_years'].sum()
    incidence_rate = (total_events / total_person_time) * 10
    print(f"total Incidence rate per person-10years: {incidence_rate}")

    TopTertile['person_years'] = TopTertile['day_difference'] / 365.25
    total_events = TopTertile['status'].sum()
    total_person_time = TopTertile['person_years'].sum()
    incidence_rate = (total_events / total_person_time) * 10
    print(f"Top Incidence rate per person-10years: {incidence_rate}")
    print(total_events)
    print(total_person_time)

    MiddleTertile['person_years'] = MiddleTertile['day_difference'] / 365.25
    total_events = MiddleTertile['status'].sum()
    total_person_time = MiddleTertile['person_years'].sum()
    incidence_rate = (total_events / total_person_time) * 10
    print(f"Middle Incidence rate per person-10years: {incidence_rate}")
    print(total_events)
    print(total_person_time)

    BottomTertile['person_years'] = BottomTertile['day_difference'] / 365.25
    total_events = BottomTertile['status'].sum()
    total_person_time = BottomTertile['person_years'].sum()
    incidence_rate = (total_events / total_person_time) * 10
    print(f"Bottom Incidence rate per person-10years: {incidence_rate}")
    print(total_events)
    print(total_person_time)

In [ ]:
PCEcatnum=1
df2=df[df['PCEcat']==PCEcatnum]
kmcurve(df2)

In [ ]:
PCEcatnum=2
df2=df[df['PCEcat']==PCEcatnum]
kmcurve(df2)

# NRI, c-index

In [ ]:
df_final=df.copy()
or statusname in ['status']:
    print('============',statusname,'============')
    NRI=0
    for i in [0,1]:
        if i==0:
            print('neg')
        else:
            print('pos')
        dftemp=df_final[df_final[statusname]==i]
        print('total',len(dftemp))

        GPT_low_PCE_low=dftemp[dftemp['PCE_ECG_class']==1]
        GPT_low_PCE_low=GPT_low_PCE_low[GPT_low_PCE_low['PCEclass']==1]
        GPT_low_PCE_mod=dftemp[dftemp['PCE_ECG_class']==1]
        GPT_low_PCE_mod=GPT_low_PCE_mod[GPT_low_PCE_mod['PCEclass']==2]
        GPT_hig_PCE_low=dftemp[dftemp['PCE_ECG_class']==2]
        GPT_hig_PCE_low=GPT_hig_PCE_low[GPT_hig_PCE_low['PCEclass']==1]
        GPT_hig_PCE_mod=dftemp[dftemp['PCE_ECG_class']==2]
        GPT_hig_PCE_mod=GPT_hig_PCE_mod[GPT_hig_PCE_mod['PCEclass']==2]
        
        totalsum=len(GPT_low_PCE_low)+len(GPT_low_PCE_mod)+len(GPT_hig_PCE_low)+len(GPT_hig_PCE_mod)
        
        print('PCEECG_low_PCE_low',len(GPT_low_PCE_low))
        print('PCEECG_low_PCE_mod',len(GPT_low_PCE_mod),len(GPT_low_PCE_mod)/totalsum)
        print('PCEECG_hig_PCE_low',len(GPT_hig_PCE_low),len(GPT_hig_PCE_low)/totalsum)
        print('PCEECG_hig_PCE_mod',len(GPT_hig_PCE_mod))
        
        print('totalsum',totalsum)
        if i==1:
            print('positive NRI',(len(GPT_hig_PCE_low)-len(GPT_low_PCE_mod))/totalsum)
            NRI+=(len(GPT_hig_PCE_low)-len(GPT_low_PCE_mod))/totalsum
        else:
            print('negative NRI',(len(GPT_low_PCE_mod)-len(GPT_hig_PCE_low))/totalsum)
            NRI+=(len(GPT_low_PCE_mod)-len(GPT_hig_PCE_low))/totalsum

    print(NRI)

In [ ]:
df_final['PCE_AI']=df_final['PCE']+df_final['AI_prediction_average']*10
print(concordance_index(df_final['day_difference'], -df_final['PCE'], df_final['status']))
print(concordance_index(df_final['day_difference'], -df_final['PCE_AI'], df_final['status']))

# characteristics table

In [ ]:
AIcat=[]
for i in range(len(df)):
    if df['AI_prediction_average'][i]>=thres2:
        AIcat.append(3)
    elif df['AI_prediction_average'][i]>=thres1:
        AIcat.append(2)
    else:
        AIcat.append(1)
df['AIcat']=AIcat

In [ ]:
statcols=[]
statcols.append('year_difference')
statcols.append('sexint')
statcols.append('DM')
statcols.append('HTN')
statcols.append('smoking')
statcols.append('SBP')
statcols.append('Chol')
statcols.append('HDL')
statcols.append('AI_prediction_average')
statcols.append('PCE')
statcols.append('status')
statcols.append('AIcat')


catcols=[]
catcols.append('sexint')
catcols.append('DM')
catcols.append('HTN')
catcols.append('smoking')
catcols.append('status')
catcols.append('AIcat')

In [ ]:
df_stats=df[statcols]

In [ ]:
def table1(catcolumn,catcolumns,dfdfdf2):
    nonnormallist=[]
    def _normality(self, x):
        #print(x.name)

        if len(x.values[~np.isnan(x.values)]) >= 20:
            p = stats.shapiro(x.values).pvalue
        else:
            p = None
        # dropna=False argument in pivot_table does not function as expected
        # return -1 instead of None
        if pd.isnull(p):
            return -1
        if p<=0.05:
            nonnormallist.append(x.name)
        return p

    TableOne._normality=_normality

    def my_custom_test(group1, group2):
        """
        Hypothesis test for test_self_defined_statistical_tests
        """
        my_custom_test.__name__ = "mannwhitneyu"
        _, pval= scipy.stats.mannwhitneyu(group1, group2)
        return pval

    nonnormallist=[]
    def _normality(self, x):
        #print(x.name)

        if len(x.values[~np.isnan(x.values)]) >= 20:
            p = stats.shapiro(x.values).pvalue
        else:
            p = None
        # dropna=False argument in pivot_table does not function as expected
        # return -1 instead of None
        if pd.isnull(p):
            return -1
        if p<=0.05:
            nonnormallist.append(x.name)
        return p

    TableOne._normality=_normality

    table1=TableOne(dfdfdf2,categorical=catcolumns,groupby=[catcolumn],normal_test=True,pval=True,htest_name=True,decimals=3)
    nonnormallist=list(set(nonnormallist))
    nonnormallist


    table1=TableOne(dfdfdf2,categorical=catcolumns,groupby=[catcolumn],normal_test=True,pval=True,htest_name=True,nonnormal=nonnormallist,decimals=3)
    try:
        os.mkdir(newtablename)
    except:
        pass
    try:
        os.mkdir(newtablename+'/table1')
    except:
        pass
    catcolumn=catcolumn.replace(' ','_')
    catcolumn=catcolumn.replace('/','_')
    table1.to_html('figures/20241001_SH_table1_'+catcolumn+'.html')

In [ ]:
table1('AIcat',catcols,df_stats)